##### Copyright 2024 Google LLC。

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 使用 DeepMind CALM 和 Gemma 進行模型組合

歡迎閱讀關於fine-tuning [Gemma](https://huggingface.co/google/gemma-2b) 使用 [Hugging Face Transformers](https://huggingface.co/docs/transformers/en/index) 和 DeepMind 的 **CALM（組合到增強語言模型）** @@P0055
隨著大型語言模型 (LLM) 變得越來越大、能力越來越強，將其擴展或適應新領域或任務可能既具有挑戰性，又成本高昂。許多解決方案涉及在新資料上重新訓練或fine-tuning大型通用模型，這是一個耗時且資源密集的過程。此外，組織限製或資料隱私問題可能會限制對此類適應所需的原始訓練資料的存取。
[**CALM**](https://github.com/google-deepmind/calm) 透過組合兩種不同的語言模型（具有基礎功能的「錨定」模型和專門針對特定領域的「增強」模型）來解決這些挑戰，而無需完全重新訓練錨定模型。 CALM 透過在模型之間引入交叉注意力來實現這一點，使您能夠結合它們的優勢並保留它們的原始功能。結果是一個更強大的組合模型，它利用現有的、經過驗證的模型和一些附加參數，而不是從頭開始建立新的整體模型。 library 目前支援組合使用 Gemma 架構建構的任兩個模型。
[**Transformers**](https://huggingface.co/docs/transformers/en/index) 是一個強大且多功能的工具，可用於處理各種大型語言模型、tokenizer 和管道。它提供了一個用戶友好的API，用於加載、訓練和部署最先進的模型，使其成為機器學習和自然語言處理生態系統不可或缺的組成部分。其廣泛的兼容性、易用性和豐富的文件有助於簡化 fine-tuning、inference 和模型評估等任務。
[**Gemma**](https://ai.google.dev/gemma) 是 Google 推出的一系列輕量級、最先進的開放模型，採用與創建 Gemini 模型相同的研究和技術構建。它們是文字到文字、僅限解碼器的大型語言模型，提供英文版本，具有開放權重、預訓練變體和指令調整變體。 Gemma 模型非常適合各種文本生成任務，包括問答、摘要和推論。它們的尺寸相對較小，因此可以將它們部署在資源有限的環境中，例如筆記型電腦、桌上型電腦或您自己的雲端基礎設施，從而實現對最先進人工智慧模型的民主化訪問，並幫助促進每個人的創新。
在此 notebook 中，您將學習如何使用 `gemma-2-2b` 模型作為錨定模型和增強模型來微調組合 LLM (CALM) 設定。產生的組合模型合併了 `gemma-2-2b` 兩個實例的功能。您也可以嘗試其他組合（`9B` 與 `2B`），但只需使用 `2B` Gemma 變體即可保持簡單。
你將學到什麼：1. **安裝和依賴關係**：安裝和導入必要的庫。
2. **模型和設定**：使用 `gemma-2-2b` 作為錨點和增強模型來初始化 CALM 設定。
3. **資料載入與預處理**：使用指令調整dataset、token進行調整，為fine-tuning做好準備。
4. **訓練**：設定訓練參數並使用Hugging Face `Trainer` 執行fine-tuning。
5. **保存與結論**：儲存微調後的模型。

<table align="left"> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/.archive/Gemma/[Gemma_2]Finetune_with_CALM.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td>
</table>

## 設定

### 選擇執行時環境

首先，您可以選擇 **Google Colab** 作為您的平台。
- #### **Google Colab** <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/d/d0/Google_Colaboratory_SVG_Logo.svg/1200px-Google_Colaboratory_SVG_Logo.svg.png" alt="Google Colab" width="30"/>

  1. 按一下「**在 Colab** 中開啟」。
  2. 您需要存取具有足夠資源的 [**Colab Pro/Pro+**](https://colab.research.google.com/signup) runtime 來執行 Gemma 模型。
  3. In the menu, go to **Runtime** > **Change runtime type**.
  4. 確保 **GPU** 設定為 **A100**。

### Gemma 使用Hugging Face

在深入學習本教學之前，讓我們先設定Gemma：
1. **建立一個 Hugging Face 帳戶**：如果您沒有帳戶，您可以[此處]註冊一個免費帳戶(https://huggingface.com/join)。
2. **造訪Gemma型號**：造訪[Gemma型號頁面](https://huggingface.com/collections/google/gemma-2-release-667d6600fd5220e7b967f315)並接受使用條件。
3. **產生Hugging Facetoken**：前往您的Hugging Face [設定頁面](https://huggingface.com/settings/tokens)並產生新的存取token（最好具有`write`權限）。在本教學的後面部分，您將需要這個token。

**完成這些步驟後，您就可以進入下一部分，在 Colab 環境中設定環境變數。 **

### 設定您的憑證

要存取私有模型和dataset，您需要登入Hugging Face（HF）生態系統。
- #### **Google Colab** <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/d/d0/Google_Colaboratory_SVG_Logo.svg/1200px-Google_Colaboratory_SVG_Logo.svg.png" alt="Google Colab" width="30"/>
如果您使用 Colab，您可以使用 Colab Secrets manager 安全地儲存 Hugging Face token (`HF_TOKEN`)：  1. 開啟 Google Colab notebook 並點選左側面板中的 🔑 Secrets 標籤。 <img src="https://storage.googleapis.com/generativeai-downloads/images/secrets.jpg" alt="The Secrets tab is found on the left panel." width=50%>
  2. **新增Hugging Facetoken**：
- 建立一個新的secret，其**名稱**為`HF_TOKEN`。 - 將token 金鑰複製/貼上到`HF_TOKEN` 的**值** 輸入框中。 - **切換**左側的按鈕以允許notebook 訪問secret。

In [ ]:
import os
import sys

if 'google.colab' in sys.modules:
    # Running on Colab
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
else:
    # Not running on Colab
    raise EnvironmentError('This notebook is designed to run on Google Colab.')

# Disable tokenizers parallelism to avoid deadlocks
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

### 設定環境

接下來，您將透過安裝 fine-tuning Gemma 模型所需的所有 Python 軟體包來設定環境。

In [ ]:
# Clone DeepMind CALM
!git clone https://github.com/google-deepmind/calm.git
%cd calm

您將複製 **CALM** 儲存庫並安裝 `transformers`、`datasets` 和 `accelerate` 的相容版本。
**注意**：您使用固定版本來確保相容性。當有新的更新可用時，您可以調整它們。

In [ ]:
# Install the appropriate Hugging Face libraries to ensure compatibility with the Gemma 2 model and CALM.
!pip install transformers==4.47.0 -U -q
!pip install datasets==3.2.0 -U -q
!pip install accelerate==1.2.1 -U -q

## 導入庫


您可以在此處匯入所需的庫，以載入和預處理 [Abirate/english_quotes](https://huggingface.co/datasets/Abirate/english_quotes) dataset、token 化、模型設定和初始化實用程式。

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModel
)
# Import the "calm" module from the "model" package for inference
from model import calm

## 使用 CALM 進行微調

在本節中，您將：1. 從 Hugging Face dataset 中載入一個小的 dataset (`Abirate/english_quotes`)。
2. 透過指定錨（基礎）模型和增強模型來設定 CALM。在此示範中，您將為兩者使用相同的 `gemma-2-2b` 模型。然而，在實踐中，您可以選擇不同的變體（例如`9B`、`27B`）來組合不同的功能。
3. 預處理 dataset 以進行語言建模。
4. 使用Hugging Face Transformers 中的`Trainer` 來微調組合模型。

為了清楚起見，您將把訓練邏輯保存到單獨的 Python 腳本 (`train.py`) 中。

### 訓練腳本

下面的腳本：- 設定 CALM 模型設定。
- 載入並token化dataset。
- 定義訓練參數並執行訓練。
- 保存微調後的模型。

您將透過命令列標誌指定 `anchor_model_dir`、`aug_model_dir`、`num_heads`、`num_connections` 等參數以及其他超參數。您可以輕鬆地針對不同的實驗調整這些標誌。

In [ ]:
%%writefile train.py
from collections.abc import Sequence
from absl import app
from absl import flags
from absl import logging

import datasets

# Import the "calm" module from the "model" package.
# This presumably contains the CALM model implementation
from model import calm

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

# Register the custom CALMConfig class under the identifier "calm" with AutoConfig.
# By doing this, when you specify a configuration type as "calm", AutoConfig knows
# to use calm.CALMConfig to instantiate the configuration.
AutoConfig.register("calm", calm.CALMConfig)

# Register the CALM model class with AutoModel for the CALMConfig configuration class.
# This means that if AutoModel is given a CALMConfig, it knows to instantiate calm.CALM.
AutoModel.register(calm.CALMConfig, calm.CALM)

_ANCHOR_MODEL_DIR = flags.DEFINE_string('anchor_model_dir', None, 'Path to the anchor model directory or identifier.')
_AUG_MODEL_DIR = flags.DEFINE_string('aug_model_dir', None, 'Path to the augmentation model directory or identifier.')
_OUTPUT_DIR = flags.DEFINE_string('output_dir', None, 'Directory where the fine-tuned model will be saved.')
_LEARNING_RATE = flags.DEFINE_float('learning_rate', 2e-5, 'Learning rate for fine-tuning.')
_EPOCHS = flags.DEFINE_integer('epochs', 3, 'Number of training epochs.')
_BATCH_SIZE = flags.DEFINE_integer('batch_size', 1, 'Batch size per device.')
_NUM_HEADS = flags.DEFINE_integer('num_heads', 1, 'Number of cross-attention heads in CALM.')
_NUM_CONNECTIONS = flags.DEFINE_integer('num_connections', 2, 'Number of cross-connections between anchor and aug models.')
_LOGGING_STEPS = flags.DEFINE_integer('logging_steps', 1, 'Logging frequency in steps.')
_MAX_STEPS = flags.DEFINE_integer('max_steps', -1, 'Max training steps, use -1 for no limit.')

def train(argv: Sequence[str]) -> None:
    del argv  # Unused.
    SEED = 42

    anchor_model_path = _ANCHOR_MODEL_DIR.value
    aug_model_path = _AUG_MODEL_DIR.value
    num_heads = _NUM_HEADS.value
    num_connections = _NUM_CONNECTIONS.value

    logging.info('Using anchor model: %s', anchor_model_path)
    logging.info('Using augmentation model: %s', aug_model_path)

    # Load the tokenizer from the anchor model
    logging.info('Loading Tokenizer...')
    tokenizer = AutoTokenizer.from_pretrained(anchor_model_path)
    tokenizer.padding_side = 'right'

    # Create CALM config
    logging.info('Creating CALM configuration...')
    calm_config = calm.CALMConfig(
        anchor_model=anchor_model_path,
        aug_model=aug_model_path,
        anchor_config=None,
        aug_config=None,
        num_connections=num_connections,
        num_heads=num_heads,
    )
    calm_config.save_pretrained('./calm_config')

    # Initialize the composed CALM model
    logging.info('Initializing the CALM model...')
    model = calm.CALM(calm_config)
    model.config.use_cache = False

    # Load the dataset (english_quotes)
    logging.info('Loading and preparing dataset...')
    dataset = datasets.load_dataset('Abirate/english_quotes', split='all')

    # Filter out empty quotes
    dataset = dataset.filter(lambda x: len(x["quote"]) > 0)

    # For demonstration, use a small subset (e.g., 2048 samples)
    dataset = dataset.shuffle(seed=SEED).select(range(2048))

    # Tokenize the data
    def preprocess_function(examples):
        return tokenizer(
            examples['quote'], truncation=True, padding='max_length',
            max_length=512
        )

    dataset = dataset.map(preprocess_function, batched=True)

    # Data collator for language modeling (no masking since it's causal LM)
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer, mlm=False
    )

    epochs = _EPOCHS.value
    batch_size = _BATCH_SIZE.value
    learning_rate = _LEARNING_RATE.value
    output_dir = _OUTPUT_DIR.value
    logging_steps = _LOGGING_STEPS.value
    max_steps = _MAX_STEPS.value

    # Split into train/validation sets
    dataset = dataset.train_test_split(test_size=0.02)

    # TrainingArguments for Hugging Face Trainer
    training_args = TrainingArguments(
        output_dir=output_dir,
        save_strategy='no',
        overwrite_output_dir=True,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        eval_strategy='epoch',
        optim="adamw_torch_fused",
        lr_scheduler_type="constant",
        warmup_ratio=0.03,
        logging_steps=logging_steps,
        max_steps=max_steps,
        learning_rate=learning_rate,
        report_to="none",
        seed=SEED,
    )

    # Initialize the Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset['train'],
        eval_dataset=dataset['test'],
        data_collator=data_collator,
        tokenizer=tokenizer,
    )

    # Train the model
    logging.info('Starting training...')
    trainer.can_return_loss = True
    trainer.train()
    trainer.save_model(output_dir)
    print(f'Training complete! Model saved to {output_dir}')

if __name__ == '__main__':
    app.run(train)

Overwriting train.py


### 開始fine-tuning

現在您可以微調組合模型。為此，您將進行僅 50 個步驟的簡短訓練以進行演示。對於真正的訓練作業，請考慮增加`max_steps` 或`epochs` 並使用更大的dataset。

In [ ]:
anchor_model_path = 'google/gemma-2-2b'
aug_model_path = 'google/gemma-2-2b'

# Remove previous output directory if exists
!rm -rf ./gemma-ft

# Run training with specified parameters
!python train.py --anchor_model_dir google/gemma-2-2b \
          --aug_model_dir google/gemma-2-2b \
          --num_heads 2 \
          --num_connections 2 \
          --learning_rate 3e-5 \
          --batch_size 2 \
          --max_steps 50 \
          --output_dir './gemma-ft'

2024-12-17 14:15:13.357550: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-17 14:15:13.375429: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-17 14:15:13.396556: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-17 14:15:13.403038: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-17 14:15:13.418338: I tensorflow/core/platform/cpu_feature_guar

## 提示使用新微調的模型

最後，讓我們使用微調後的模型prompt，並驗證它是否真的按預期工作。為此，我們首先使用 tokenizer 產生輸入 ID，然後使用 `model.generate()` 依靠重新加載的微調模型來產生回應，從而使用範例 prompt 來測試模型。

In [ ]:
# Register the custom CALMConfig and CALM classes with AutoConfig and AutoModel
AutoConfig.register("calm", calm.CALMConfig)
AutoModel.register(calm.CALMConfig, calm.CALM)

In [ ]:
# Load the CALM configuration
config = calm.CALMConfig.from_pretrained('./calm_config')

# Load the composed and fine-tuned model
model = calm.CALM.from_pretrained('./gemma-ft', config=config)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

CALM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

CALM(
  (anchor_model): Gemma2ForCausalLM(
    (model): Gemma2Model(
      (embed_tokens): Embedding(256000, 2304, padding_idx=0)
      (layers): ModuleList(
        (0-25): 26 x Gemma2DecoderLayer(
          (self_attn): Gemma2Attention(
            (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
            (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
            (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
            (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
            (rotary_emb): Gemma2RotaryEmbedding()
          )
          (mlp): Gemma2MLP(
            (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
            (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
            (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
            (act_fn): PytorchGELUTanh()
          )
          (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
   

In [ ]:
print('Loading Tokenizer...')
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('google/gemma-2-2b', use_fast=True)
tokenizer.padding_side = 'right'

print('Prompting the model...')
prompt = "Life is either a "
inputs = tokenizer(prompt, return_tensors='pt').to(device)
outputs = model.generate(**inputs, max_new_tokens=40, use_cache=False,
                         repetition_penalty=1.1)
text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(text)

Loading Tokenizer...
Prompting the model...
Life is either a <strong>journey</strong> or a <strong>destination.</strong> If it's the former, you'll never arrive; if it's the latter, you'll never depart.

- Anonymous




您已使用 `gemma-2-2b` 作為錨定模型和增強模型成功微調了 CALM 組合模型。雖然本演示側重於一個簡單的小型範例，但對於較大的模型和datasets，其原理仍然相同。透過遵循本指南，您學習如何使用 CALM 組合兩個 Gemma 模型，以建立一個整合兩個模型功能的新模型，擴展其技能，而不會產生完全重新訓練的計算開銷。
### 後續步驟：
- 嘗試不同的 Gemma 模型變體或其他指令調整模型。
- 使用更大的datasets 和更多的訓練步驟來獲得更好的模型品質。
- 調整超參數（例如學習率、批量大小、時期）以獲得最佳結果。

快樂fine-tuning！